In [5]:
#!/usr/bin/env python3
"""
Debug script: compare LHP channel selection between Script 1 and Script 2
region mask logic for a given subject/session.
"""

import numpy as np
import pandas as pd
import cmlreaders as cml

# ============================================================================
# REGION MASK — SCRIPT 1 logic (label-prefix + broad region search)
# ============================================================================

def define_region_masks_script1(pairs_df):
    """
    Script 1 logic:
    - Searches multiple region columns in priority order
    - Hemisphere inferred first from channel label prefix (L/R),
      then falls back to region string containing 'left'/'right'
    - Hippocampus: substring match (case-insensitive)
    - LTC: STG / MTG / ITG / Superior/Middle/Inferior Temporal
    """
    n = len(pairs_df)
    masks = {}

    # Find region column
    region_col = None
    for col_candidate in ['mni.region', 'ind.region', 'stein.region',
                          'avg.region', 'dk.region']:
        if col_candidate in pairs_df.columns:
            region_col = col_candidate
            break
    if region_col is None:
        region_cols = [c for c in pairs_df.columns if 'region' in c.lower()]
        region_col  = region_cols[0] if region_cols else None

    if region_col is None:
        print("  [Script 1] ERROR: No region column found!")
        for r in ['LHP', 'RHP']:
            masks[r] = np.zeros(n, dtype=bool)
        return masks, None

    regions_str = pairs_df[region_col].fillna('')

    def is_left(row_idx):
        if 'label' in pairs_df.columns:
            lbl = str(pairs_df['label'].iloc[row_idx])
            if lbl and lbl[0].upper() == 'L':
                return True
            if lbl and lbl[0].upper() == 'R':
                return False
        return 'left' in regions_str.iloc[row_idx].lower()

    left_mask  = np.array([is_left(i) for i in range(n)])
    right_mask = ~left_mask

    hipp_mask    = regions_str.str.contains('Hippocampus', case=False, na=False).values
    masks['LHP'] = hipp_mask & left_mask
    masks['RHP'] = hipp_mask & right_mask

    return masks, region_col


# ============================================================================
# REGION MASK — SCRIPT 2 logic (regex on single preferred column)
# ============================================================================

def define_region_masks_script2(pairs_df):
    """
    Script 2 logic:
    - Searches single preferred column (dk > mni > ind > first available)
    - Uses full Left/Right regex: 'Left.*Hippocampus', 'Right.*Hippocampus'
    - No label-prefix hemisphere fallback
    """
    masks = {}

    if 'dk.region' in pairs_df.columns:
        col = 'dk.region'
    elif 'mni.region' in pairs_df.columns:
        col = 'mni.region'
    elif 'ind.region' in pairs_df.columns:
        col = 'ind.region'
    else:
        region_cols = [c for c in pairs_df.columns if 'region' in c.lower()]
        col = region_cols[0] if region_cols else None

    if col is None:
        print("  [Script 2] ERROR: No region column found!")
        for r in ['LHP', 'RHP']:
            masks[r] = np.zeros(len(pairs_df), dtype=bool)
        return masks, None

    masks['LHP'] = pairs_df[col].str.contains('Left.*Hippocampus',  case=False, na=False, regex=True)
    masks['RHP'] = pairs_df[col].str.contains('Right.*Hippocampus', case=False, na=False, regex=True)

    return masks, col


# ============================================================================
# COMPARISON
# ============================================================================

def compare_lhp_masks(subject, experiment, session):
    print(f"\n{'='*70}")
    print(f"Subject: {subject}  |  Experiment: {experiment}  |  Session: {session}")
    print(f"{'='*70}")

    reader   = cml.CMLReader(subject, experiment, session=session)
    pairs_df = reader.load('pairs')

    print(f"\nTotal bipolar pairs: {len(pairs_df)}")

    # Show available region columns
    region_cols_present = [c for c in pairs_df.columns if 'region' in c.lower()]
    print(f"Region columns present: {region_cols_present}")

    # Run both mask functions
    masks_s1, col_s1 = define_region_masks_script1(pairs_df)
    masks_s2, col_s2 = define_region_masks_script2(pairs_df)

    print(f"\nScript 1 used region column : {col_s1}")
    print(f"Script 2 used region column : {col_s2}")
    same_col = col_s1 == col_s2
    print(f"Same column?                : {'YES' if same_col else 'NO — different columns already a source of divergence'}")

    # ── Per-hemisphere comparison ────────────────────────────────────────────
    for region in ['LHP', 'RHP']:
        mask1 = masks_s1[region]
        mask2 = masks_s2[region]

        chs1 = set(pairs_df[mask1]['label'].tolist()) if 'label' in pairs_df.columns else set(pairs_df[mask1].index.tolist())
        chs2 = set(pairs_df[mask2]['label'].tolist()) if 'label' in pairs_df.columns else set(pairs_df[mask2].index.tolist())

        only_s1 = sorted(chs1 - chs2)
        only_s2 = sorted(chs2 - chs1)
        shared  = sorted(chs1 & chs2)

        print(f"\n── {region} ──────────────────────────────────────────────────────")
        print(f"  Script 1 : {len(chs1)} channels  {sorted(chs1)}")
        print(f"  Script 2 : {len(chs2)} channels  {sorted(chs2)}")
        print(f"  Shared   : {len(shared)}  {shared}")
        print(f"  Only S1  : {len(only_s1)}  {only_s1}")
        print(f"  Only S2  : {len(only_s2)}  {only_s2}")

        if only_s1 or only_s2:
            print(f"\n  *** MISMATCH — inspecting divergent channels ***")
            divergent_labels = set(only_s1) | set(only_s2)
            if 'label' in pairs_df.columns:
                div_df = pairs_df[pairs_df['label'].isin(divergent_labels)].copy()
            else:
                div_df = pairs_df.iloc[list(mask1 ^ mask2)].copy()

            display_cols = ['label'] if 'label' in pairs_df.columns else []
            if col_s1 and col_s1 in pairs_df.columns:
                display_cols.append(col_s1)
            if col_s2 and col_s2 != col_s1 and col_s2 in pairs_df.columns:
                display_cols.append(col_s2)

            # Add hemisphere diagnosis
            if 'label' in pairs_df.columns:
                div_df = div_df.copy()
                div_df['label_prefix'] = div_df['label'].str[0].str.upper()
                if col_s1:
                    div_df['region_says_left'] = div_df[col_s1].str.lower().str.contains('left', na=False)
                display_cols += ['label_prefix']
                if col_s1:
                    display_cols.append('region_says_left')

            print(f"\n  Divergent channel details:")
            print(div_df[display_cols].to_string(index=True))
        else:
            print(f"  ✓ Perfect agreement — both scripts select identical {region} channels")

    # ── Summary ──────────────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print("SUMMARY")
    print(f"{'='*70}")
    for region in ['LHP', 'RHP']:
        n1 = int(masks_s1[region].sum())
        n2 = int(masks_s2[region].sum())
        agreement = n1 == n2 and set(
            pairs_df[masks_s1[region]]['label'].tolist() if 'label' in pairs_df.columns else []
        ) == set(
            pairs_df[masks_s2[region]]['label'].tolist() if 'label' in pairs_df.columns else []
        )
        status = "✓ match" if agreement else f"✗ MISMATCH  (S1={n1}, S2={n2})"
        print(f"  {region}: {status}")


# ============================================================================
# BATCH MODE — run across all sessions for an experiment
# ============================================================================

def batch_compare(experiment, max_subjects=None):
    print(f"\n{'#'*70}")
    print(f"BATCH COMPARISON — {experiment}")
    print(f"{'#'*70}")

    whole_df = cml.CMLReader.get_data_index()
    exp_df   = whole_df[whole_df['experiment'] == experiment]
    subjects = sorted(exp_df['subject'].unique())

    if max_subjects:
        subjects = subjects[:max_subjects]

    summary_rows = []

    for subject in subjects:
        for session in exp_df[exp_df['subject'] == subject]['session'].unique():
            try:
                reader   = cml.CMLReader(subject, experiment, session=int(session))
                pairs_df = reader.load('pairs')

                masks_s1, col_s1 = define_region_masks_script1(pairs_df)
                masks_s2, col_s2 = define_region_masks_script2(pairs_df)

                for region in ['LHP', 'RHP']:
                    chs1 = set(pairs_df[masks_s1[region]]['label'].tolist()) if 'label' in pairs_df.columns else set()
                    chs2 = set(pairs_df[masks_s2[region]]['label'].tolist()) if 'label' in pairs_df.columns else set()
                    summary_rows.append({
                        'subject':    subject,
                        'session':    session,
                        'region':     region,
                        'n_s1':       len(chs1),
                        'n_s2':       len(chs2),
                        'n_shared':   len(chs1 & chs2),
                        'n_only_s1':  len(chs1 - chs2),
                        'n_only_s2':  len(chs2 - chs1),
                        'col_s1':     col_s1,
                        'col_s2':     col_s2,
                        'match':      chs1 == chs2,
                    })
            except Exception as e:
                print(f"  FAILED {subject} sess={session}: {e}")

    summary = pd.DataFrame(summary_rows)
    if summary.empty:
        print("No results.")
        return summary

    print("\nOverall match rate by region:")
    print(summary.groupby('region')['match'].mean().map(lambda x: f"{x*100:.1f}%"))

    mismatches = summary[~summary['match']]
    print(f"\nMismatched sessions: {len(mismatches)} / {len(summary)}")
    if not mismatches.empty:
        print(mismatches[['subject','session','region','n_s1','n_s2',
                           'n_only_s1','n_only_s2','col_s1','col_s2']].to_string(index=False))

    return summary


# ============================================================================
# ENTRY POINT
# ============================================================================

if __name__ == '__main__':

    # ── Single session deep-dive ─────────────────────────────────────────────
    # Replace with a real subject/session from your data index
    SUBJECT    = 'R1524T'
    EXPERIMENT = 'DBOY1'
    SESSION    = 0

    compare_lhp_masks(SUBJECT, EXPERIMENT, SESSION)

    # ── Batch across all DBOY1 sessions ──────────────────────────────────────
    # summary = batch_compare('DBOY1')
    # summary.to_csv('region_mask_comparison.csv', index=False)


Subject: R1524T  |  Experiment: DBOY1  |  Session: 0

Total bipolar pairs: 226
Region columns present: ['avg.region', 'avg.corrected.region', 'hcp.region', 'ind.region', 'ind.corrected.region', 'mni.region', 'stein.region', 'vox.region']

Script 1 used region column : mni.region
Script 2 used region column : mni.region
Same column?                : YES

── LHP ──────────────────────────────────────────────────────
  Script 1 : 0 channels  []
  Script 2 : 0 channels  []
  Shared   : 0  []
  Only S1  : 0  []
  Only S2  : 0  []
  ✓ Perfect agreement — both scripts select identical LHP channels

── RHP ──────────────────────────────────────────────────────
  Script 1 : 0 channels  []
  Script 2 : 0 channels  []
  Shared   : 0  []
  Only S1  : 0  []
  Only S2  : 0  []
  ✓ Perfect agreement — both scripts select identical RHP channels

SUMMARY
  LHP: ✓ match
  RHP: ✓ match
